# 05 - Nonlinear models

Same cold-start prediction task as notebook 04, but with tree-based models
(Random Forest, Gradient Boosting) that can learn interaction effects - e.g.
trusting a participant's own early slope more when they have more early
observations (`has_own_slope`, `{domain}_n_obs`).

In [1]:
import sys
sys.path.append("..")

import os
import pandas as pd

from src.preprocessing import load_data, truncate_to_early_visits, validate_longitudinal
from src.longitudinal.slope_extraction import extract_slopes
from src.longitudinal.trends import build_trend_features
from src.models.models import build_feature_matrix, build_rf_model, build_gbr_model, get_naive_baseline
from src.evaluation.evaluation import evaluate_cross_participant


In [2]:
DOMAINS = ("memory", "attention", "language")

sim_df = validate_longitudinal(load_data("../data/simulated/longitudinal_simulated.csv"))
true_slopes_df = pd.read_csv("../data/simulated/true_slopes.csv", dtype={"participant_id": str})

early_df = truncate_to_early_visits(sim_df, n_visits=2)
early_slope_table = extract_slopes(early_df, domains=DOMAINS)
early_trend_table = build_trend_features(early_df, domains=DOMAINS)


c:\Users\yusef\Downloads\Cognitive-Decline-Prediction-main\.venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


c:\Users\yusef\Downloads\Cognitive-Decline-Prediction-main\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\yusef\Downloads\Cognitive-Decline-Prediction-main\.venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2206: ConvergenceWarning: MixedLM optimization failed, trying a different optimizer may help.
  warnings.warn(msg, ConvergenceWarning)
c:\Users\yusef\Downloads\Cognitive-Decline-Prediction-main\.venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2218: ConvergenceWarning: Gradient optimization failed, |grad| = 99.541350
  warnings.warn(msg, ConvergenceWarning)


c:\Users\yusef\Downloads\Cognitive-Decline-Prediction-main\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\yusef\Downloads\Cognitive-Decline-Prediction-main\.venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2206: ConvergenceWarning: MixedLM optimization failed, trying a different optimizer may help.
  warnings.warn(msg, ConvergenceWarning)
c:\Users\yusef\Downloads\Cognitive-Decline-Prediction-main\.venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2218: ConvergenceWarning: Gradient optimization failed, |grad| = 130.438346
  warnings.warn(msg, ConvergenceWarning)
c:\Users\yusef\Downloads\Cognitive-Decline-Prediction-main\.venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2261: ConvergenceWarning: The Hessian matrix at the estimated parameter values is not positive defi

In [3]:
models = {
    "baseline_mean": get_naive_baseline(),
    "random_forest": build_rf_model(),
    "gbr": build_gbr_model(),
}

nonlinear_results = []
for domain in DOMAINS:
    X, y, groups = build_feature_matrix(early_slope_table, early_trend_table, true_slopes_df, domain=domain)
    result = evaluate_cross_participant(X, y, groups, models)
    result["domain"] = domain
    nonlinear_results.append(result)

nonlinear_results_df = pd.concat(nonlinear_results, ignore_index=True)
nonlinear_results_df


,model,mode,mae,rmse,r2,n,domain
0,baseline_mean,cross_participant,0.060712,0.075489,-0.004900,250,memory
1,random_forest,cross_participant,0.055148,0.069319,0.152657,250,memory
2,gbr,cross_participant,0.059370,0.073905,0.036832,250,memory
3,baseline_mean,cross_participant,0.041327,0.051553,-0.001703,250,attention
4,random_forest,cross_participant,0.044151,0.054065,-0.101721,250,attention
5,gbr,cross_participant,0.047592,0.058046,-0.269913,250,attention
6,baseline_mean,cross_participant,0.035957,0.043218,-0.004245,250,language
7,random_forest,cross_participant,0.034833,0.043172,-0.002132,250,language
8,gbr,cross_participant,0.035825,0.045052,-0.091313,250,language


In [4]:
os.makedirs("../data/processed", exist_ok=True)
nonlinear_results_df.to_csv("../data/processed/nonlinear_model_results.csv", index=False)
